# 章节实践

完成FastGelu算子的完整开发，在regbase模式下实现算子，并应用Double Buffer性能优化。

**公式**:

$$y = \frac{x}{1 + e^{-x}}$$

**要求**:
- 使用 regbase 模式实现 FastGelu 算子的完整开发（Host 侧 + Kernel 侧）
- 实现 FastGelu 的 VF 函数 `y = x / (1 + exp(-1.702 * x))`
- 应用 Double Buffer（BUFFER_NUM = 2）优化流水线
- 处理数据量 = 16384 个 float，8 核并行

请开始你的实践，体验从理解到创造的完整开发过程。

---

## 环境准备


In [ ]:
!mkdir -p Sources/03.07

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("Environment initialization process completed successfully!")


---

## 头文件和常量

首先编写头文件和常量定义。本实践使用 regbase 模式，包含必要的系统头文件和 AscendC 算子开发头文件。

| 符号 | 值 | 说明 |
|------|-----|------|
| `BUFFER_NUM` | 2 | 双缓冲深度，实现流水线并行 |
| `TOTAL_LENGTH` | 8 x 2048 = 16384 | 总数据量（float） |
| `TILE_NUM` | 8 | 每核分块数 |
| `BLOCK_DIM` | 8 | 并行 AI Core 数量 |

`FastGeluArgs` 结构体用于将 Host 侧参数传递给 Kernel 侧（总长度和分块数）。


In [ ]:
%%writefile Sources/03.07/fast_gelu.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <cmath>
#include "acl/acl.h"
#include "kernel_operator.h"

using namespace AscendC;
using namespace AscendC::Reg;

constexpr uint32_t BUFFER_NUM = 2;
constexpr uint32_t TOTAL_LENGTH = 8 * 2048;
constexpr uint32_t TILE_NUM = 8;
constexpr uint32_t BLOCK_DIM = 8;

struct FastGeluArgs {
    uint32_t totalLength;
    uint32_t tileNum;
};


---

## VF 函数实现

VF 函数（`__simd_vf__`）是 regbase 模式的核心，直接在向量寄存器上完成计算。
整个 VF 函数遵循固定的**四步流水线**:

```
LoadAlign -> (Neg -> Exp -> Adds -> Div) -> StoreAlign -> LocalMemBar
  1 Load      2 Compute chain         3 Store     4 Barrier
```

| 步骤 | 操作 | 说明 |
|------|------|------|
| 1 Load | `LoadAlign` | 从 UB 加载 vecLen 个元素到寄存器 |
| 2 Neg | `Neg(tReg, xReg)` | 取反: t = -x |
| 2 Exp | `Exp(tReg, tReg)` | 指数: t = exp(-x) |
| 2 Adds | `Adds(tReg, tReg, 1.0f)` | 加标量: t = 1 + exp(-x) |
| 2 Div | `Div(tReg, xReg, tReg)` | 相除: t = x / (1 + exp(-x)) |
| 3 Store | `StoreAlign` | 结果寄存器写回 UB |
| 4 Barrier | `LocalMemBar` | 保证 Store 完成后再 Load（多 tile 迭代必需） |

关键参数:
- `vecLen = GetVecLen() / sizeof(float)` - 单次向量操作元素数
- `repeat = tileLength / vecLen` - 循环加载次数
- `MaskReg mask = CreateMask<T, MaskPattern::ALL>()` - 全开掩码


In [ ]:
%%writefile -a Sources/03.07/fast_gelu.asc

// FastGelu VF: y = x / (1 + exp(-1.702 * x))
template <typename T>
__simd_vf__ inline void FastGeluVF(__ubuf__ T* xAddr, __ubuf__ T* yAddr,
                                     uint32_t vecLen, uint32_t repeat) {
    MaskReg mask = CreateMask<T, MaskPattern::ALL>();
    RegTensor<T> xReg, tReg;

    for (uint16_t i = 0; i < repeat; ++i) {
        LoadAlign(xReg, xAddr + i * vecLen);
        Neg(tReg, xReg, mask);
        Exp(tReg, tReg, mask);
        Adds(tReg, tReg, 1.0f, mask);
        Div(tReg, xReg, tReg, mask);
        StoreAlign(yAddr + i * vecLen, tReg, mask);
    }
    LocalMemBar<MemType::VEC_STORE, MemType::VEC_LOAD>();
}


---

## Kernel 实现（Double Buffer）

Kernel 侧使用 Double Buffer 优化（BUFFER_NUM = 2），通过乒乓缓冲掩盖搬运延迟。
整个处理流程为 **CopyIn -> Compute -> CopyOut** 三段式流水线：

| 阶段 | 方法 | 操作 |
|------|------|------|
| Init | 分核 + 队列分配 | blockLength = TOTAL_LENGTH / BLOCK_DIM; 绑定 GlobalTensor; InitBuffer 分配队列 |
| Process | 流水循环 | loopCount = TILE_NUM x BUFFER_NUM; 迭代调用 CopyIn/Compute/CopyOut |
| CopyIn | GM -> UB | AllocTensor -> DataCopy -> EnQue |
| Compute | UB -> Reg -> UB | DeQue -> AllocTensor -> FastGeluVF -> EnQue -> FreeTensor |
| CopyOut | UB -> GM | DeQue -> DataCopy -> FreeTensor |

**Double Buffer 原理**:
- BUFFER_NUM = 2 时，队列深度为 2，可实现搬运与计算的重叠
- CopyIn 写入 queue[0] 的同时，Compute 读取 queue[1]
- 总迭代次数 = TILE_NUM x BUFFER_NUM，每次处理 tileLength 个元素


In [ ]:
%%writefile -a Sources/03.07/fast_gelu.asc

class KernelFastGelu {
public:
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y) {
        blockLength = TOTAL_LENGTH / BLOCK_DIM;
        tileLength = blockLength / TILE_NUM / BUFFER_NUM;
        vecLen = GetVecLen() / sizeof(float);
        uint32_t offset = blockLength * GetBlockIdx();
        xGm.SetGlobalBuffer((__gm__ float*)x + offset, blockLength);
        yGm.SetGlobalBuffer((__gm__ float*)y + offset, blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, tileLength * sizeof(float));
        pipe.InitBuffer(outQueueY, BUFFER_NUM, tileLength * sizeof(float));
    }

    __aicore__ inline void Process() {
        for (int32_t i = 0; i < TILE_NUM * BUFFER_NUM; i++) {
            CopyIn(i); Compute(i); CopyOut(i);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress) {
        LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
        DataCopy(xLocal, xGm[progress * tileLength], tileLength);
        inQueueX.EnQue(xLocal);
    }

    __aicore__ inline void Compute(int32_t progress) {
        LocalTensor<float> xLocal = inQueueX.DeQue<float>();
        LocalTensor<float> yLocal = outQueueY.AllocTensor<float>();
        FastGeluVF<float>(
            reinterpret_cast<__ubuf__ float*>(xLocal.GetPhyAddr()),
            reinterpret_cast<__ubuf__ float*>(yLocal.GetPhyAddr()),
            vecLen, tileLength / vecLen);
        outQueueY.EnQue<float>(yLocal);
        inQueueX.FreeTensor(xLocal);
    }

    __aicore__ inline void CopyOut(int32_t progress) {
        LocalTensor<float> yLocal = outQueueY.DeQue<float>();
        DataCopy(yGm[progress * tileLength], yLocal, tileLength);
        outQueueY.FreeTensor(yLocal);
    }

    TPipe pipe;
    TQue<TPosition::VECIN, BUFFER_NUM> inQueueX;
    TQue<TPosition::VECOUT, BUFFER_NUM> outQueueY;
    GlobalTensor<float> xGm, yGm;
    uint32_t blockLength, tileLength, vecLen;
};

__global__ __aicore__ void fast_gelu_kernel(GM_ADDR x, GM_ADDR y,
                                              FastGeluArgs args) {
    KernelFastGelu op;
    op.Init(x, y);
    op.Process();
}


---

## Host 主函数

Host 侧负责完整的 ACL 生命周期管理：初始化 -> 内存分配 -> 数据准备 -> 核函数调用 -> 结果验证 -> 资源释放。

| 阶段 | ACL API | 说明 |
|------|---------|------|
| 初始化 | `aclInit` / `aclrtSetDevice` / `aclrtCreateStream` | 启动 ACL 运行时 |
| 内存分配 | `aclrtMallocHost` / `aclrtMalloc` | Host 和 Device 端申请内存 |
| 数据准备 | for 循环初始化 xHost, aclrtMemcpy H2D | 生成 -4.0 ~ +4.0 输入数据 |
| 调用 | `<<<BLOCK_DIM, stream>>>` | 启动 8 核并行 Kernel |
| 同步 | `aclrtSynchronizeStream` | 等待 Kernel 执行完成 |
| 回拷 | aclrtMemcpy D2H | 将 Device 结果拷回 Host |
| 验证 | for 循环 + fabsf 比对 | CPU 参考值: x/(1+exp(-x)), 容差 1e-4f |
| 释放 | aclrtFree / aclrtFreeHost | 释放所有资源 |
| 结束 | `aclrtDestroyStream` / `aclrtResetDevice` / `aclFinalize` | 关闭 ACL 运行时 |


In [ ]:
%%writefile -a Sources/03.07/fast_gelu.asc

int32_t main() {
    aclInit(nullptr);
    aclrtSetDevice(0);
    aclrtStream stream;
    aclrtCreateStream(&stream);

    float *xHost, *yHost;
    aclrtMallocHost((void**)&xHost, TOTAL_LENGTH * sizeof(float));
    aclrtMallocHost((void**)&yHost, TOTAL_LENGTH * sizeof(float));
    for (uint32_t i = 0; i < TOTAL_LENGTH; i++) {
        xHost[i] = -4.0f + i * 8.0f / TOTAL_LENGTH;
    }

    void *xDev, *yDev;
    aclrtMalloc(&xDev, TOTAL_LENGTH * sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc(&yDev, TOTAL_LENGTH * sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(xDev, TOTAL_LENGTH * sizeof(float), xHost,
                TOTAL_LENGTH * sizeof(float), ACL_MEMCPY_HOST_TO_DEVICE);

    FastGeluArgs args = {TOTAL_LENGTH, TILE_NUM};
    fast_gelu_kernel<<<BLOCK_DIM, nullptr, stream>>>(xDev, yDev, args);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(yHost, TOTAL_LENGTH * sizeof(float), yDev,
                TOTAL_LENGTH * sizeof(float), ACL_MEMCPY_DEVICE_TO_HOST);

    bool pass = true;
    for (uint32_t i = 0; i < TOTAL_LENGTH; i++) {
        if (fabsf(yHost[i] - xHost[i] / (1.0f + expf(-xHost[i]))) > 1e-4f) {
            pass = false; break;
        }
    }
    printf("FastGelu Complete: %s\n", pass ? "PASSED" : "FAILED");

    aclrtFree(xDev); aclrtFree(yDev);
    aclrtFreeHost(xHost); aclrtFreeHost(yHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(0);
    aclFinalize();
    return 0;
}


---

## 编译运行

使用 bisheng 编译器将 `.asc` 源码编译为可执行文件。

> **注意**: 编译时需指定 `--npu-arch=dav-2201` 架构参数。


In [ ]:
!bisheng Sources/03.07/fast_gelu.asc --npu-arch=dav-2201 -o Sources/03.07/fast_gelu


In [ ]:
!./Sources/03.07/fast_gelu


预期输出：`FastGelu Complete: PASSED`

---

## 答案参考

完成实践后，可以参考以下答案代码。


In [ ]:
!cat ./answer/03.07_answer/fast_gelu_complete.asc
